In [1]:
from math_datasets.datasets import Dataset, GSM8K, SVAMP
from math_datasets.evaluator import evaluate_all, evaluate_detail
from pathlib import Path
from math_datasets.generators import ReWOOGenerate
import os

/opt/miniconda3/envs/MA312/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


MPS (Metal Performance Shaders) is available! Using Apple Silicon GPU.


In [13]:
model = "qwen3:0.6b"
SAVE_DIR = Path(os.getcwd())
def get_ollama_model_name_identifer(model_name: str) -> str:
    return "ollama/" + model_name

model_identifier_run_name = get_ollama_model_name_identifer(model)
ds = GSM8K

In [14]:
df_detail = evaluate_detail(model_identifier_run_name, ds, SAVE_DIR.as_posix(), use_transformated_answers=False, additional_metrics=ReWOOGenerate)

In [15]:
df_detail.head()

,question,answer,model_history,response,is_correct,target_answer,format_correct,input_tokens,output_tokens,total_tokens,reasoning
0,Janet’s ducks lay 16 eggs per day. She eats th...,Janet sells 16 - 3 - 4 = <<16-3-4=9>>9 duck eg...,Error occured.,Error occured.,False,Janet sells 16 - 3 - 4 = <<16-3-4=9>>9 duck eg...,False,NaN,NaN,NaN,None
1,A robe takes 2 bolts of blue fiber and half th...,It takes 2/2=<<2/2=1>>1 bolt of white fiber\nS...,[{'plan': {'steps': [['Calculate the white fib...,3.0,True,It takes 2/2=<<2/2=1>>1 bolt of white fiber\nS...,True,258.0,460.0,718.0,"<think>\nOkay, let's see. The problem is about..."
2,Josh decides to try flipping a house. He buys...,The cost of the house and repairs came out to ...,[{'plan': {'steps': [['Calculate the new house...,40000.0,False,The cost of the house and repairs came out to ...,True,289.0,474.0,763.0,"<think>\nOkay, let's see. The problem is about..."
3,James decides to run 3 sprints 3 times a week....,He sprints 3*3=<<3*3=9>>9 times\nSo he runs 9*...,Error occured.,Error occured.,False,He sprints 3*3=<<3*3=9>>9 times\nSo he runs 9*...,False,NaN,NaN,NaN,None
4,"Every day, Wendi feeds each of her chickens th...","If each chicken eats 3 cups of feed per day, t...",[{'plan': {'steps': [['Calculate per chicken f...,180.0,False,"If each chicken eats 3 cups of feed per day, t...",True,343.0,1024.0,1367.0,"<think>\nOkay, let's see. The problem is about..."


In [16]:
df_detail.to_csv(SAVE_DIR / f"{ds.name}_detail.csv", index=False)

## Correct Analysis

In [6]:
# number correct
df_detail["is_correct"].value_counts(normalize=True)

is_correct
False    0.69
True     0.31
Name: proportion, dtype: float64

In [7]:
# True: in x % of answers, when the format was correct it provided the right answer.
df_detail[df_detail["format_correct"] == True]["is_correct"].value_counts(normalize=True)

is_correct
True     0.775
False    0.225
Name: proportion, dtype: float64

## Format Analysis

In [8]:
df_detail["format_correct"].value_counts(normalize=True)

format_correct
False    0.6
True     0.4
Name: proportion, dtype: float64

In [9]:
# Format Correct Score where the end result is false
## If the score for False is high this means that if the answer is incorrect, then it is most likely due to incorrect format
df_detail[df_detail["is_correct"] == False]["format_correct"].value_counts(normalize=True)

format_correct
False    0.869565
True     0.130435
Name: proportion, dtype: float64

## Token Usage

In [10]:
df_detail["input_tokens"].mean(), df_detail["output_tokens"].mean(), df_detail["total_tokens"].mean()

(np.float64(271.2), np.float64(413.825), np.float64(685.025))

In [11]:
df_format_correct = df_detail[df_detail["format_correct"] == True]
df_correct = df_format_correct[df_format_correct["is_correct"] == True]

df_correct["input_tokens"].mean(), df_correct["output_tokens"].mean(), df_correct["total_tokens"].mean()

(np.float64(271.0),
 np.float64(413.96774193548384),
 np.float64(684.9677419354839))

In [12]:
df_wrong = df_format_correct[df_format_correct["is_correct"] == False]
df_wrong["input_tokens"].mean(), df_wrong["output_tokens"].mean(), df_wrong["total_tokens"].mean()

(np.float64(271.8888888888889),
 np.float64(413.3333333333333),
 np.float64(685.2222222222222))